# Phase 1 - Plant disease detection model (MobileNetV2 on PlantVillage)

Capstone Project 7, Group 4. This notebook trains, evaluates and publishes the disease classifier
described in the proposal (Section 9, Phase 1) and hands it over to Phase 5.

**What it does**

1. Downloads PlantVillage (color) from Kaggle via `kagglehub`.
2. Fine-tunes `google/mobilenet_v2_1.0_224` with augmentation for tilted / inverted / blurry photos.
3. Reports accuracy, per-class precision / recall / F1, confusion matrix, top-3 and a confidence-threshold sweep.
4. Tests on real-world internet images (target: at least 50% correct).
5. Pushes the model, reports and model card to **https://huggingface.co/neghani/plantvillage-mobilenetv2** and tags `v1.0`.

All logic lives in `model/*.py` in the repository; this notebook only runs those scripts.

**How to test (local vs Colab)**

- **Local (CPU):** edit `model/*.py`, smoke-test tomato / 40 images / 1 epoch, then `git push`.
- **Colab (GPU):** do **not** `files.upload()` code. Re-run the sync cell below. It always
  replaces tracked files with `origin/main` (`notebooks/colab_sync.py`) so you never re-test a stale upload.
- **Real-world images:** git-ignored. Keep a zip on Drive and pass `--real-world-from` (always replaces).

**Before you start (Colab)**

- Runtime -> Change runtime type -> **T4 GPU** (or better).
- Colab secrets (key icon in the left bar): add `HF_TOKEN` (a *write* token from https://huggingface.co/settings/tokens).
  Optional: `KAGGLE_USERNAME` and `KAGGLE_KEY` if `kagglehub` asks for them (the dataset is public, so usually not needed).

In [ ]:
!nvidia-smi -L || echo "No GPU - training will be very slow; switch the runtime to a GPU."

## 1. Sync the repository (always replace from GitHub)

Fetches `notebooks/colab_sync.py` from GitHub, then hard-resets tracked files to `origin/main`.
Checkpoints in `model/outputs/` and images under `model/data/real_world/` are left alone.

After you push a local fix, **re-run this cell** (or the one-file cell after it). Do not upload `.py` files.

In [ ]:
import os, urllib.request
from pathlib import Path

REPO = "neghani/iiith-group4-capstone-plat-leaf-decease-detection"
RAW = f"https://raw.githubusercontent.com/{REPO}/main/notebooks/colab_sync.py"
helper = Path("/tmp/colab_sync.py")
try:
    urllib.request.urlretrieve(RAW, helper)
    print("Fetched", RAW)
except Exception as exc:
    local = Path("notebooks/colab_sync.py")
    if not local.exists():
        raise RuntimeError("Could not fetch colab_sync.py from GitHub; clone the repo first.") from exc
    helper = local
    print("Using local", helper, "because raw fetch failed:", exc)
%run $helper
os.chdir(os.environ.get("REPO_ROOT", os.getcwd()))
print("cwd", os.getcwd())

Optional: replace **one** file after a local push (still from GitHub, always overwrite). Change the path and re-run. Do not `files.upload()`.

In [ ]:
# Re-run after `git push`. Always overwrites the listed file(s) from origin/main.
# Uncomment extra --file flags as needed.
!python notebooks/colab_sync.py --skip-install --file model/train.py
# !python notebooks/colab_sync.py --skip-install --file model/evaluate.py --file model/predict.py --file model/common.py

## 2. Log in to Hugging Face

The push target is fixed to `neghani/plantvillage-mobilenetv2`. If you run this with a different account, set `HF_MODEL_ID` to your own namespace below.

In [ ]:
import os
from huggingface_hub import HfApi, login

try:
    from google.colab import userdata  # type: ignore
    token = userdata.get("HF_TOKEN")
except Exception:
    token = os.environ.get("HF_TOKEN")
assert token, "Add HF_TOKEN to Colab secrets (enable 'Notebook access') or export HF_TOKEN."
login(token=token, add_to_git_credential=False)
os.environ["HF_TOKEN"] = token

HF_MODEL_ID = os.environ.get("HF_MODEL_ID", "neghani/plantvillage-mobilenetv2")
me = HfApi().whoami()["name"]
expected_owner = HF_MODEL_ID.split("/")[0]
assert me == expected_owner or expected_owner in [o["name"] for o in HfApi().whoami().get("orgs", [])], (
    f"Logged in as '{me}' but HF_MODEL_ID is '{HF_MODEL_ID}'. "
    f"Either log in as '{expected_owner}' or set os.environ['HF_MODEL_ID'] = '{me}/plantvillage-mobilenetv2' before this cell."
)
os.environ["HF_MODEL_ID"] = HF_MODEL_ID
print("Logged in as", me, "-> will push to", HF_MODEL_ID)

## 3. Download PlantVillage (color)

In [ ]:
import kagglehub, os
root = kagglehub.dataset_download("abdallahalidev/plantvillage-dataset")
print("Downloaded to", root)

import sys
sys.path.insert(0, "model")
from common import resolve_data_dir
DATA_DIR = str(resolve_data_dir(root))
os.environ["DATA_DIR"] = DATA_DIR
classes = sorted(d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d)))
print(len(classes), "classes, e.g.", classes[:5])
print("images:", sum(len(os.listdir(os.path.join(DATA_DIR, c))) for c in classes))

## 4. Quick smoke test (optional, ~2 min)

Tomato classes only, 60 images per class, 1 epoch. Confirms the pipeline before spending GPU time
on the full run. Nothing is pushed.

In [ ]:
!python model/train.py \
  --data_dir "$DATA_DIR" --crop_filter Tomato --max_samples_per_class 60 \
  --output_dir model/outputs/smoke \
  --do_train --do_eval --num_train_epochs 1 \
  --per_device_train_batch_size 32 --per_device_eval_batch_size 64 \
  --learning_rate 5e-4 --fp16 --report_to none --save_strategy no --logging_steps 5
!python model/evaluate.py --model model/outputs/smoke

## 5. Full training run

All 38 classes (~54k images). On a T4 one epoch is roughly 8-10 minutes with the settings below;
5 epochs is a good first full run. The best checkpoint by validation macro-F1 is kept, the test
split is scored, and everything is pushed to the Hub as soon as training finishes
(`--push_to_hub`), so a Colab disconnect after this cell does not lose the model.

To try the ResNet fallback from the proposal, change `--model_name_or_path microsoft/resnet-50`
and `--output_dir model/outputs/plantvillage-resnet50` (and `--hub_model_id` if you want a
separate repo).

In [ ]:
import os
OUTPUT_DIR = "model/outputs/plantvillage-mobilenetv2"
os.environ["OUTPUT_DIR"] = OUTPUT_DIR
!python model/train.py \
  --model_name_or_path google/mobilenet_v2_1.0_224 \
  --data_dir "$DATA_DIR" \
  --output_dir "$OUTPUT_DIR" \
  --do_train --do_eval \
  --num_train_epochs 5 \
  --per_device_train_batch_size 64 --per_device_eval_batch_size 128 \
  --learning_rate 5e-4 --weight_decay 0.01 --warmup_steps 0.05 --lr_scheduler_type cosine \
  --fp16 --dataloader_num_workers 2 \
  --eval_strategy epoch --save_strategy epoch --save_total_limit 2 \
  --load_best_model_at_end --metric_for_best_model macro_f1 --greater_is_better True \
  --logging_steps 50 --report_to none --seed 42 \
  --push_to_hub --hub_model_id "$HF_MODEL_ID" --hub_strategy end

## 6. Evaluate on the held-out test split and build the model card

Produces `reports/` (metrics.json, classification report, confusion matrix, threshold sweep) and fills `README.md`; `--push` uploads both to the Hub.

In [ ]:
!python model/evaluate.py --model "$OUTPUT_DIR" --hub_model_id "$HF_MODEL_ID" --push

In [ ]:
from IPython.display import Image, display
import json, os
display(Image(filename=os.path.join(os.environ["OUTPUT_DIR"], "reports", "confusion_matrix.png"), width=900))
m = json.load(open(os.path.join(os.environ["OUTPUT_DIR"], "reports", "metrics.json")))
print({k: m[k] for k in ("accuracy", "macro_f1", "top3_accuracy", "recommended_confidence_threshold")})

## 7. Real-world images (target: at least 50% top-1)

Collect internet photos into `model/data/real_world/<ClassName>/` (see
`model/data/real_world/README.md`). Images are git-ignored.

**Do not `files.upload()` every run.** Put `real_world.zip` on Drive once
(`MyDrive/plant-disease/real_world.zip`). The next cell always deletes existing class folders
and copies from that zip (or a local folder).

In [ ]:
# Always replace class folders from Drive (change the path if yours differs).
# Zip layout: <ClassName>/*.jpg  or  real_world/<ClassName>/*.jpg
REAL_WORLD_SRC = "/content/drive/MyDrive/plant-disease/real_world.zip"
!python notebooks/colab_sync.py --skip-install --real-world-from $REAL_WORLD_SRC

In [ ]:
!python model/real_world_eval.py --model "$OUTPUT_DIR" --hub_model_id "$HF_MODEL_ID" --push

## 8. Publish and tag the release for Phase 5

`model/publish.py` uses the `HF_TOKEN` Colab secret: it checks the token and owner, uploads the final model folder
(weights, `config.json`, `splits.json`, `reports/`, model card; no checkpoints) and tags `v1.0`.
Run the dry run first if you want to see the file list.

Phase 5 pins `HF_MODEL_REVISION=v1.0` so later retraining cannot silently change the API's behaviour.

In [ ]:
!python model/publish.py --model "$OUTPUT_DIR" --hub_model_id "$HF_MODEL_ID" --dry_run

In [ ]:
!python model/publish.py --model "$OUTPUT_DIR" --hub_model_id "$HF_MODEL_ID" --tag v1.0

## 9. Inference demo from the Hub (what Phase 5 will do)

In [ ]:
import json, os, random
from PIL import Image as PILImage
from IPython.display import display

import sys
sys.path.insert(0, "model")
from predict import diagnose

m = json.load(open(os.path.join(os.environ["OUTPUT_DIR"], "reports", "metrics.json")))
threshold = m["recommended_confidence_threshold"]

data_dir = os.environ["DATA_DIR"]
cls = random.choice(sorted(os.listdir(data_dir)))
img_path = os.path.join(data_dir, cls, random.choice(os.listdir(os.path.join(data_dir, cls))))
display(PILImage.open(img_path).resize((224, 224)))
print("True class:", cls)
print(json.dumps(diagnose(img_path, model_id=os.environ["HF_MODEL_ID"], revision="v1.0", threshold=threshold), indent=2))

## 10. Handover to Phase 5

Put these in `.env` of the API service:

```env
HF_MODEL_ID=neghani/plantvillage-mobilenetv2
HF_MODEL_REVISION=v1.0
CONFIDENCE_THRESHOLD=<recommended_confidence_threshold from reports/metrics.json>
```

- `model/predict.py` is the reference implementation (quality gate -> model -> top-k -> not sure).
- Class names in `config.json` (`id2label`) are PlantVillage folder names and match the keys of
  `api/data/knowledge/diseases.json`.
- Full instructions, challenges and the ResNet fallback are in `model/README.md`.